# Use case: Climate DT ocean surface over the Arctic

Earth Data Hub offers an innovative and super-efficient way to access data.

Here we present how to access the Destination Earth Climate Digital Twin (Climate DT) ocean data and combine three variables on a single map of the Northern Hemisphere, seen from above the North Pole: the sea surface temperature, the sea ice concentration and the sea surface height. Note that we will be using the IFS-NEMO daily ocean dataset of the SSP3-7.0 scenario at high resolution, which contains several global 2D ocean variables.

## Setup the environment

**If you haven't done it already, follow the [Getting started notebook](./00-getting-started.ipynb) to setup your environment and DestinE credentials.**

**Note:** the Climate DT datasets require upgraded access. See the [Earth Data Hub documentation](https://earthdatahub.destine.eu/docs/data-access.html#upgraded-access) to find out how to apply.

This notebook also needs the optional map dependencies, _cartopy_ and _cmasher_. If you are using `uv` as in the README, start Jupyter Lab with `uv run --frozen --extra maps jupyter lab`.

In [ ]:
import cartopy.crs as ccrs
import cmasher
import fsspec
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import LightSource

## Choose what to plot

We define:
* the day of interest,
* the area of interest, the whole Northern Hemisphere, as `[W, E, S, N]`,
* the map projection, an orthographic view from above the North Pole,
* the local cache directory, see the [local cache notebook](./60-best-practice-local-cache-era5-small-area.ipynb) for details.

In [ ]:
time = "2015-01-01"
extent = [-180, 180, 0, 90]  # W, E, S, N
projection = ccrs.Orthographic(central_longitude=-35, central_latitude=90)
cache_storage = "/tmp/edh_cache"

## Access the Climate DT data

The best practice for accessing data from the Earth Data Hub comprises the following steps:
1. open the dataset using the code snippet found on its dataset page in the [Earth Data Hub catalogue](https://earthdatahub.destine.eu/)
2. select the area of interest
3. select the time of interest

### Open the dataset and select the data

The following assumes you set up your EDH API key in your _netrc_ file.

We add the `simplecache` directive, so the data chunks are cached on the local file system and running the notebook again doesn't download the data nor consume your EDH request quota.

Note that all operations are lazy, that is they are not applied to the whole dataset, but are just recorded for later use. Download and computations are only done when requested, in our case when plotting.

In [ ]:
url = "https://api.earthdatahub.destine.eu/climate-dt-2/IFS-NEMO-SSP3-7.0-o2d-daily-high-v0.zarr"
ds = xr.open_dataset(
    f"asyncwrapper::simplecache::{url}",
    storage_options={
        "https": {"client_kwargs": {"trust_env": True}},
        "simplecache": {"cache_storage": f"{cache_storage}/climate-dt"},
    },
    chunks={},
    engine="zarr",
)
ds = ds.sel(longitude=slice(*extent[:2]), latitude=slice(*extent[2:]), time=time)
ds

We use three variables:
* `avg_tos`, the daily mean sea surface temperature, that we convert from Kelvin to °C,
* `avg_siconc`, the daily mean sea ice concentration, a fraction between 0 and 1,
* `avg_zos`, the daily mean sea surface height, whose small variations are related to ocean currents and eddies.

In [ ]:
sst = ds["avg_tos"] - 273.15
sst.attrs = {"long_name": "sea surface temperature", "units": "°C"}
siconc = ds["avg_siconc"]
ssh = ds["avg_zos"]

## Prepare the land background

We use the _Gray Earth_ shaded relief raster from [Natural Earth](https://www.naturalearthdata.com/), a zipped GeoTIFF. The `simplecache` and `zip` directives of _fsspec_ download it once and then read it from the local cache.

We open it with Xarray, select the area of interest (latitudes run from north to south) and rename the coordinates to match the Climate DT ones.

In [ ]:
raster = fsspec.open_local(
    "simplecache::zip://GRAY_50M_SR_W.tif::https://naturalearth.s3.amazonaws.com/50m_raster/GRAY_50M_SR_W.zip",
    simplecache={"cache_storage": f"{cache_storage}/natural-earth"},
)
land = xr.open_dataarray(raster, engine="rasterio", chunks={}).squeeze()
land = land.sel(x=slice(*extent[:2]), y=slice(extent[3], extent[2]))
land = land.rename(x="longitude", y="latitude")

## Plot the ocean surface

We draw four layers with `imshow`, from bottom to top:
1. the land, with the Gray Earth shaded relief,
2. the sea surface temperature, with the _cmasher_ `pride` colormap, from -2°C, about the freezing point of sea water, to 30°C,
3. the sea ice, as white with transparency given by the concentration: no ice is fully transparent, full coverage is opaque,
4. the sea surface height, as a hillshade with a strong vertical exaggeration, so that its small variations become visible as a relief.

The `get_extent` helper returns the `[W, E, S, N]` extent of each layer, as expected by `imshow`.

**This operation is the slow one, as this is when the data is actually downloaded.** Thanks to local caching, running it again is much faster.

In [ ]:
def get_extent(da):
    lon, lat = da["longitude"], da["latitude"]
    return [lon.min().item(), lon.max().item(), lat.min().item(), lat.max().item()]


kwargs = {"extent": get_extent(ds), "transform": ccrs.PlateCarree(), "origin": "lower"}

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"projection": projection})

# 1. Land
ax.imshow(land, extent=get_extent(land), transform=ccrs.PlateCarree(), cmap="binary")

# 2. Sea surface temperature
im = ax.imshow(sst, vmin=-2, vmax=30, cmap=cmasher.pride, **kwargs)

# 3. Sea ice
ice_rgba = np.ones((*siconc.shape, 4))
ice_rgba[..., 3] = siconc.fillna(0).clip(0, 1)
ax.imshow(ice_rgba, **kwargs)

# 4. Sea surface height
ssh_shade = LightSource().hillshade(ssh.fillna(0).values, vert_exag=1_000)
ax.imshow(ssh_shade, alpha=0.15, cmap="Greys", **kwargs)

fig.colorbar(im, ax=ax, shrink=0.7, extend="both", label="sea surface temperature [°C]")
ax.set_title(f"Climate DT IFS-NEMO SSP3-7.0, {time}");

### Next steps

* change `time` to any other day of the dataset, e.g. a day in September, when the sea ice extent is at its minimum,
* look at the Southern Hemisphere with `extent = [-180, 180, -90, 0]` and `central_latitude=-90`.